# Nirbhay: AI-Powered Digital Safety & Response Platform
### End-Semester Major Project Demonstration | Department of Data Science & Machine Learning
**Navrachana University, Vadodara, Gujarat (2026-27)**  
**Guide:** Prof. Ujwala Patigara  
**Presented by:** Isha Rajput, Parisha Parmar, Priya Bhatt  

---
### Live Demonstration Objectives (Faculty Requirements):
1. **Running of Model:** Multimodal inference on multilingual messages (English, Hindi, Gujarati) and URLs with real-time risk scoring and plain-language explainability.
2. **Evaluating Model with Live Data:** Live test data evaluation, confusion matrices, Precision, Recall, F1-Score, and False Positive Rate (FPR) per language, plus comparative ablation analysis (Text-Only vs. URL-Only vs. Multi-Evidence Fusion).

In [ ]:
# Step 1: Environment Setup and Imports
import sys, os
BASE_DIR = os.path.abspath('..')
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ml.feature_extractor import detect_language, extract_url_features, FEATURE_COLUMNS
from backend.ml_engine.predictor import nirbhay_engine

print("Nirbhay DS-ML Engine loaded successfully!")

## 1. Multilingual Dataset Inspection
Examine dataset distribution across English, Hindi (Devanagari), and Gujarati languages and class balance (Phishing vs. Legitimate).

In [ ]:
df = pd.read_csv('../data/multilingual_phishing_dataset.csv')
print(f"Total records: {len(df)}")
print("\nLanguage distribution:")
print(df['language'].value_counts())
print("\nClass balance:")
print(df['label'].value_counts())
df.head()

## 2. Pointer 1: Live Running of the Model
Test real-time parallel model inference on English, Hindi, and Gujarati phishing & legitimate samples.

In [ ]:
live_test_cases = [
    {
        "name": "Hindi SBI KYC Suspension Scam",
        "text": "प्रिय एसबीआई ग्राहक, आपका बैंक खाता और योनो ऐप आज रात 9 बजे तक बंद कर दिया जाएगा क्योंकि आपका पैन कार्ड और केवाईसी अपडेट नहीं है। तुरंत लिंक पर क्लिक करके केवाईसी पूरा करें: http://sbi-kyc-quickupdate.xyz/login",
        "url": "http://sbi-kyc-quickupdate.xyz/login"
    },
    {
        "name": "Gujarati MGVCL Electricity Power Cut Scam",
        "text": "પ્રિય ગ્રાહક, તમારા પાછલા મહિનાનું વીજળી બિલ અપડેટ ન હોવાથી આજે રાત્રે 9:30 વાગ્યે તમારું વીજ કનેક્શન કાપી નાખવામાં આવશે: http://mgvcl-online-bill-update.xyz",
        "url": "http://mgvcl-online-bill-update.xyz"
    },
    {
        "name": "English Courier Redelivery Scam",
        "text": "India Post: Your package cannot be delivered due to wrong address. Pay Rs 25 redelivery fee: http://indiapost-parcel-address-update.xyz/tracking",
        "url": "http://indiapost-parcel-address-update.xyz/tracking"
    },
    {
        "name": "Legitimate SBI Transaction Alert (English)",
        "text": "Dear Customer, INR 1,500.00 debited from A/C **4589 on 01-Nov-26 at ATM VADODARA. Avl Bal: INR 34,250.75. Not you? Visit https://onlinesbi.sbi",
        "url": "https://onlinesbi.sbi"
    }
]

for case in live_test_cases:
    res = nirbhay_engine.predict(text=case['text'], url=case['url'])
    print("=" * 75)
    print(f"TEST CASE: {case['name']}")
    print(f"Detected Language : {res['language']['name']}")
    print(f"Threat Level      : {res['threat_level']} (Score: {res['risk_score']}%)")
    print(f"Modality Scores   : Text={res['modality_scores']['text_only_risk']}% | URL={res['modality_scores']['url_only_risk']}% | Fused={res['modality_scores']['fused_risk']}%")
    print(f"Summary           : {res['explainability']['summary']}")
    print("Guidance          : " + " | ".join(res['explainability']['actionable_guidance'][:2]))

## 3. Pointer 2: Evaluating Model with Live Data
Evaluate the model against custom live test samples and compute Precision, Recall, F1-Score, and False Positive Rate (FPR).

In [ ]:
live_batch = [
    {"text": "SBI account blocked. Click http://sbi-kyc-verify-portal.top", "url": "http://sbi-kyc-verify-portal.top", "label": 1, "language": "en"},
    {"text": "તમારું વીજળી જોડાણ આજે કાપી નાખવામાં આવશે: http://mgvcl-bill-pay-online.xyz", "url": "http://mgvcl-bill-pay-online.xyz", "label": 1, "language": "gu"},
    {"text": "बधाई! आपने 25 लाख की लॉटरी जीती है: http://kbc-lucky-winner-claim.top", "url": "http://kbc-lucky-winner-claim.top", "label": 1, "language": "hi"},
    {"text": "Dear SBI Customer, INR 5,000 credited to A/C **4589. Bal: Rs 42,100.", "url": "", "label": 0, "language": "en"},
    {"text": "તમારા ખાતામાં રૂ. 4,000 જમા થયા છે. ઉપલબ્ધ બેલેન્સ રૂ. 22,150 છે.", "url": "", "label": 0, "language": "gu"},
    {"text": "प्रिय ग्राहक, आपके खाते में रु 2,500 जमा किए गए हैं। कुल शेष रु 18,900 है।", "url": "", "label": 0, "language": "hi"},
    {"text": "URGENT: Call bank manager immediately at 9823491029 to unfreeze account.", "url": "", "label": 1, "language": "en"},
    {"text": "Good morning, your requested bank account statement is ready: http://192.168.4.15:8080/kyc-update", "url": "http://192.168.4.15:8080/kyc-update", "label": 1, "language": "en"}
]

eval_res = nirbhay_engine.evaluate_live_batch(live_batch)
fm = eval_res['fusion_metrics']
print("LIVE EVALUATION METRICS ON UNSEEN DATA:")
print(f"Total Tested : {eval_res['total_evaluated']}")
print(f"Accuracy     : {fm['accuracy']*100:.2f}%")
print(f"Precision    : {fm['precision']*100:.2f}%")
print(f"Recall       : {fm['recall']*100:.2f}%")
print(f"F1-Score     : {fm['f1_score']*100:.2f}%")
print(f"FPR          : {fm['false_positive_rate']*100:.2f}%")

## 4. Ablation Study & Research Question Verification
**Research Question:** "Does combining message text and URL evidence improve scam detection compared to analyzing either source alone across English, Hindi, and Gujarati?"

Comparative metrics between **Text-Only**, **URL-Only**, and **Multi-Evidence Fusion**.

In [ ]:
report = nirbhay_engine.eval_report
abl = report['ablation_study']

abl_df = pd.DataFrame({
    "Modality": ["Text-Only NLP", "URL-Only Lexical", "Multi-Evidence Fusion"],
    "Accuracy (%)": [abl['text_only']['accuracy']*100, abl['url_only']['accuracy']*100, abl['multi_evidence_fusion']['accuracy']*100],
    "Precision (%)": [abl['text_only']['precision']*100, abl['url_only']['precision']*100, abl['multi_evidence_fusion']['precision']*100],
    "Recall (%)": [abl['text_only']['recall']*100, abl['url_only']['recall']*100, abl['multi_evidence_fusion']['recall']*100],
    "F1-Score (%)": [abl['text_only']['f1_score']*100, abl['url_only']['f1_score']*100, abl['multi_evidence_fusion']['f1_score']*100],
    "False Positive Rate (%)": [abl['text_only']['false_positive_rate']*100, abl['url_only']['false_positive_rate']*100, abl['multi_evidence_fusion']['false_positive_rate']*100]
})

print(abl_df.to_string(index=False))

# Plot Ablation Comparison
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(abl_df))
width = 0.2

ax.bar(x - width, abl_df["Accuracy (%)"], width, label="Accuracy", color="#6366f1")
ax.bar(x, abl_df["Recall (%)"], width, label="Recall", color="#3b82f6")
ax.bar(x + width, abl_df["F1-Score (%)"], width, label="F1-Score", color="#06b6d4")

ax.set_ylabel("Percentage (%)")
ax.set_title("Ablation Analysis: Text-Only vs. URL-Only vs. Multi-Evidence Fusion")
ax.set_xticks(x)
ax.set_xticklabels(abl_df["Modality"])
ax.set_ylim(80, 105)
ax.legend()
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()